# SpikeInterface Processing Pipeline for Tetrodes & Neuronexus Probe
### Jake Swann


#### Run axona_spikeinterface script on all recordings marked on a spreadsheet
- See single-recording script for more details

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) #gets rid of some annoying warnings
import pandas as pd
import numpy as np
import spikeinterface as si
import spikeinterface.extractors as se
from pathlib import Path
from utils import *
import os
import phy.apps.kwik

%matplotlib widget

sheet = pd.read_excel('probe_data_IV.xlsx')
sheet_inc = sheet[sheet['include_spikeinterface'] == 'Y']

trial_list = sheet_inc['trial_name']. to_list()
session_list = np.unique([i[:6] for i in trial_list])    

# Collect into sessions and preprocess
recording_list = []
for i,session in enumerate(session_list):
    recording_list.append([])
    for j in trial_list:
        if session in j:
            base_folder = sheet_inc[sheet_inc['trial_name'] == j]['path'].tolist()[0]
            
            recording_list[i].append([preprocess(recording = se.read_axona(f"{base_folder}/{j}.set"),
                                     recording_name = j,
                                     base_folder = base_folder,
                                     electrode_type = sheet_inc[sheet_inc['trial_name'] == j]['probe_type'].tolist()[0]),
                                     j, base_folder])
                                     

# Concatenate over a single session and sort
recordings_concat = []
for i, recording in enumerate(recording_list):
    session = pd.DataFrame(recording_list[0])
    
    recordings_concat.append(si.concatenate_recordings(session.iloc[:,0]))
    for j, concat in enumerate(recordings_concat):
        print(f'Sorting {concat}')
        sort(recording = concat, 
             recording_name = session_list[j], 
             base_folder = session.iloc[0,2], 
             sorting_suffix = 'sorting_1.5_4')
print(session)

/home/isabella/Documents/isabella/klusta_testdata/220624a_famBox_preprocessed
32-channel 32 ch four shanks recording loaded from previous preprocessing
BinaryFolderRecording: 32 channels - 1 segments - 48.0kHz - 900.062s
/home/isabella/Documents/isabella/klusta_testdata/220624b_famBox_preprocessed
32-channel 32 ch four shanks recording loaded from previous preprocessing
BinaryFolderRecording: 32 channels - 1 segments - 48.0kHz - 900.062s
/home/isabella/Documents/isabella/klusta_testdata/220624c_diffBox_preprocessed
32-channel 32 ch four shanks recording loaded from previous preprocessing
BinaryFolderRecording: 32 channels - 1 segments - 48.0kHz - 900.062s
/home/isabella/Documents/isabella/klusta_testdata/220624d_CCE_preprocessed
32-channel 32 ch four shanks recording loaded from previous preprocessing
BinaryFolderRecording: 32 channels - 1 segments - 48.0kHz - 900.062s
/home/isabella/Documents/isabella/klusta_testdata/220624e_famBox_preprocessed
32-channel 32 ch four shanks recording l

write_binary_recording:   0%|          | 0/6901 [00:00<?, ?it/s]

RUNNING SHELL SCRIPT: /home/isabella/Documents/isabella/klusta_testdata/220624_sorting_1.5_4/run_klusta.sh
18:42:02 [I] creator:446          Using kwik file `/data/isabella/klusta_testdata/220624_sorting_1.5_4/recording.kwik`.

18:42:02 [I] launch:206           Starting spike detection.

18:42:02 [I] launch:53            SpikeDetekt parameters:

18:42:02 [I] launch:55            chunk_overlap_seconds = 0.015

18:42:02 [I] launch:55            chunk_size_seconds = 3

18:42:02 [I] launch:55            connected_component_join_size = 1

18:42:02 [I] launch:55            detect_spikes = negative

18:42:02 [I] launch:55            dtype = int16

18:42:02 [I] launch:55            excerpt_size_seconds = 1

18:42:02 [I] launch:55            experiment_name = recording.kwik

18:42:02 [I] launch:55            extract_s_after = 32

18:42:02 [I] launch:55            extract_s_before = 16

18:42:02 [I] launch:55            filter_butter_order = 3

18:42:02 [I] launch:55            filter_high_facto

In [ ]:
#Run phy on klusta output
os.environ["QTWEBENGINE_CHROMIUM_FLAGS"] = "--single-process"

## If tetrodes sorted separately, change channel_group to choose tetrode to display in phy
phy.apps.kwik.kwik_gui(sorting_path/'recording.kwik', channel_group=None, clustering=None)